# 02 – Factor Analysis (FA)

**PM2.5 Time Series: Factor Analysis & SARIMA**  
Notebook này triển khai **Step 2 – Factor Analysis** dựa trên đúng logic trong `src/factor_analysis.py`.

## Mục tiêu

- Chọn các biến môi trường/khí tượng (không dùng `PM2.5`) để rút gọn chiều.
- Chuẩn hoá dữ liệu (z-score).
- Kiểm định mức độ phù hợp cho FA (KMO, Bartlett).
- Scree plot (eigenvalues) và chọn số nhân tố theo Kaiser (λ > 1).
- Fit Factor Analysis (method=`principal`, rotation=`varimax`).
- Xuất:
  - `reports/figures/02_fa/scree_plot.png`
  - `reports/figures/02_fa/factor_loadings_heatmap.png`
  - `reports/tables/factor_loadings.csv`
  - `reports/tables/factor_interpretation.txt`
  - `data/processed/fa_data.csv` (dữ liệu gốc + Factor1..k)

## Điều kiện trước khi chạy

- Đã có `data/interim/cleaned_data.csv` (tạo từ Step 1 / `01_EDA.ipynb` hoặc `python src/data_prep.py`).

In [ ]:
# Thiết lập môi trường
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from factor_analyzer import FactorAnalyzer
from factor_analyzer.factor_analyzer import calculate_bartlett_sphericity, calculate_kmo
from sklearn.preprocessing import StandardScaler

# Project root
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
INTERIM_DIR = ROOT / "data" / "interim"
PROCESSED_DIR = ROOT / "data" / "processed"
TABLES_DIR = ROOT / "reports" / "tables"
FIG_DIR = ROOT / "reports" / "figures" / "02_fa"

for p in [INTERIM_DIR, PROCESSED_DIR, TABLES_DIR, FIG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

plt.rcParams["figure.dpi"] = 100
plt.rcParams["savefig.dpi"] = 300
pd.set_option("display.max_columns", 50)

print("ROOT:", ROOT)
print("cleaned_data:", INTERIM_DIR / "cleaned_data.csv")

## 1. Đọc dữ liệu đã làm sạch

Nguồn dữ liệu cho FA là `data/interim/cleaned_data.csv` (đã xử lý missing và datetime index).

In [ ]:
cleaned_path = INTERIM_DIR / "cleaned_data.csv"
if not cleaned_path.exists():
    raise FileNotFoundError(f"Missing: {cleaned_path}. Hãy chạy Step 1 trước.")

df = pd.read_csv(cleaned_path, index_col=0, parse_dates=True)

print("Shape:", df.shape)
print("Index range:", df.index.min(), "->", df.index.max())
print("Index dtype:", df.index.dtype)
print("Index unique:", df.index.is_unique)

# Tần suất (nếu infer được)
try:
    inferred = pd.infer_freq(df.index)
except Exception:
    inferred = None
print("Inferred freq:", inferred)

# Kiểm tra khoảng thời gian bị thiếu theo lưới giờ (đúng đặc thù dataset PRSA hourly)
full_hourly = pd.date_range(df.index.min(), df.index.max(), freq="H")
missing_ts = full_hourly.difference(df.index)
dup_ts = df.index[df.index.duplicated()]

print("Expected hourly length:", len(full_hourly))
print("Missing timestamps (hourly grid):", len(missing_ts))
print("Duplicated timestamps:", len(dup_ts))

# Lưu báo cáo kiểm tra time index
check_path = TABLES_DIR / "time_index_check_cleaned_data.txt"
with open(check_path, "w", encoding="utf-8") as f:
    f.write("Time Index Check (cleaned_data.csv)\n")
    f.write("===============================\n\n")
    f.write(f"Index range: {df.index.min()} -> {df.index.max()}\n")
    f.write(f"Index unique: {df.index.is_unique}\n")
    f.write(f"Inferred freq: {inferred}\n\n")
    f.write(f"Expected hourly length: {len(full_hourly)}\n")
    f.write(f"Missing timestamps (hourly grid): {len(missing_ts)}\n")
    if len(missing_ts) > 0:
        f.write("First 20 missing timestamps:\n")
        for ts in missing_ts[:20]:
            f.write(f"  - {ts}\n")
        if len(missing_ts) > 20:
            f.write(f"  ... ({len(missing_ts) - 20} more)\n")
    f.write("\n")
    f.write(f"Duplicated timestamps: {len(dup_ts)}\n")
    if len(dup_ts) > 0:
        f.write("First 20 duplicated timestamps:\n")
        for ts in dup_ts[:20]:
            f.write(f"  - {ts}\n")
        if len(dup_ts) > 20:
            f.write(f"  ... ({len(dup_ts) - 20} more)\n")

print("Saved:", check_path)

df.head()

## 2. Chọn biến cho Factor Analysis (đúng như `src/factor_analysis.py`)

FA dùng nhóm biến môi trường/khí tượng, **loại `PM2.5`** (biến mục tiêu) và các biến không phù hợp (như `wd`, `station`, `No`).

Danh sách biến mặc định (theo code):

- `PM10`, `SO2`, `NO2`, `CO`, `O3`, `TEMP`, `PRES`, `DEWP`, `RAIN`, `WSPM`

In [ ]:
FA_VARS = ["PM10", "SO2", "NO2", "CO", "O3", "TEMP", "PRES", "DEWP", "RAIN", "WSPM"]

available = [c for c in FA_VARS if c in df.columns]
if len(available) < 3:
    raise ValueError(f"Need at least 3 FA variables. Found: {available}")

fa_data = df[available].copy()
print("FA variables used:", available)
fa_data.describe().T

## 3. Kiểm định tính phù hợp cho FA (KMO & Bartlett)

- **KMO**: > 0.6 thường được xem là phù hợp (càng gần 1 càng tốt).
- **Bartlett’s test**: p-value nhỏ (ví dụ < 0.05) → bác bỏ giả thuyết ma trận tương quan là đơn vị → phù hợp để làm FA.

In [ ]:
# Không dropna để tránh đứt gãy chuỗi thời gian.
# Thay vào đó: nội suy theo thời gian cho các biến numeric, rồi ffill/bfill ở biên.
fa_filled = fa_data.copy()

missing_before = fa_filled.isna().sum().sort_values(ascending=False)
print("Missing before (top):")
display(missing_before[missing_before > 0].head(10))

for col in fa_filled.columns:
    if fa_filled[col].isna().any():
        fa_filled[col] = fa_filled[col].interpolate(method="time")

fa_filled = fa_filled.ffill().bfill()

missing_after = fa_filled.isna().sum().sort_values(ascending=False)
print("Missing after (top):")
display(missing_after[missing_after > 0].head(10))

# Lưu missing summary trước/sau (FA vars)
ms_before = pd.DataFrame({"missing_count": missing_before, "missing_ratio": missing_before / len(fa_filled)})
ms_after = pd.DataFrame({"missing_count": missing_after, "missing_ratio": missing_after / len(fa_filled)})
ms_before.to_csv(TABLES_DIR / "missing_summary_fa_vars_before_interpolate.csv")
ms_after.to_csv(TABLES_DIR / "missing_summary_fa_vars_after_interpolate.csv")

# KMO/Bartlett cần dữ liệu không có NaN
kmo_all, kmo_model = calculate_kmo(fa_filled)
chi_square_value, p_value = calculate_bartlett_sphericity(fa_filled)

print(f"KMO overall: {kmo_model:.4f}")
print("KMO per variable:")
for var, val in zip(available, kmo_all):
    print(f"  - {var}: {val:.4f}")

print("\nBartlett's test of sphericity")
print(f"  Chi-square: {chi_square_value:.4f}")
print(f"  p-value:   {p_value:.6f}")

# Save text report
out_txt = TABLES_DIR / "kmo_bartlett_results.txt"
with open(out_txt, "w", encoding="utf-8") as f:
    f.write("KMO and Bartlett's Test Results for Factor Analysis\n")
    f.write("=================================================\n\n")
    f.write(f"Variables used: {', '.join(available)}\n\n")
    f.write("Missing handling for FA vars: interpolate(method='time') + ffill/bfill\n\n")
    f.write(f"KMO overall: {kmo_model:.4f}\n")
    f.write("KMO per variable:\n")
    for var, val in zip(available, kmo_all):
        f.write(f"  - {var}: {val:.4f}\n")
    f.write("\nBartlett's test of sphericity:\n")
    f.write(f"  Chi-square: {chi_square_value:.4f}\n")
    f.write(f"  p-value: {p_value:.6f}\n")

print("\nSaved:", out_txt)
print("Saved:", TABLES_DIR / "missing_summary_fa_vars_before_interpolate.csv")
print("Saved:", TABLES_DIR / "missing_summary_fa_vars_after_interpolate.csv")

## 4. Chuẩn hoá dữ liệu (z-score)

Giống `standardize()` trong `src/factor_analysis.py`: dùng `StandardScaler()` để đưa các biến về mean=0, std=1 trước khi làm FA.

In [ ]:
scaler = StandardScaler()
fa_std_values = scaler.fit_transform(fa_filled)
fa_std = pd.DataFrame(fa_std_values, index=fa_filled.index, columns=fa_filled.columns)

fa_std.describe().T.loc[:, ["mean", "std"]].head()

## 5. Scree plot (eigenvalues) và chọn số nhân tố

- Tính eigenvalues từ **ma trận tương quan** của dữ liệu chuẩn hoá.
- Vẽ **Scree plot**.
- Chọn số nhân tố theo **Kaiser criterion**: số eigenvalues > 1, giới hạn trong [2, 5] (giống `select_n_factors`).

In [ ]:
# Eigenvalues từ correlation matrix
corr = fa_std.corr()
eigenvalues, _ = np.linalg.eig(corr)
eigenvalues = np.real(eigenvalues)
eigenvalues = np.sort(eigenvalues)[::-1]

# Kaiser criterion
kaiser = int(np.sum(eigenvalues > 1))
n_factors = int(np.clip(kaiser, 2, min(5, len(eigenvalues))))

print("Eigenvalues:", np.round(eigenvalues, 3))
print(f"Kaiser count (>1): {kaiser}")
print(f"Selected n_factors (bounded [2,5]): {n_factors}")

# Scree plot (giống plot_scree)
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(range(1, len(eigenvalues) + 1), eigenvalues, color="#2E86AB", alpha=0.8, edgecolor="white")
ax.plot(range(1, len(eigenvalues) + 1), eigenvalues, "o-", color="#E94F37", linewidth=2, markersize=8)
ax.axhline(y=1, color="gray", linestyle="--", label="Kaiser criterion (λ=1)")
ax.set_xlabel("Factor Number", fontsize=11)
ax.set_ylabel("Eigenvalue", fontsize=11)
ax.set_title("Scree Plot (Factor Analysis)", fontsize=14)
ax.set_xticks(range(1, len(eigenvalues) + 1))
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()

scree_path = FIG_DIR / "scree_plot.png"
plt.savefig(scree_path, dpi=300, bbox_inches="tight")
plt.show()
print("Saved:", scree_path)

## 6. Fit Factor Analysis (principal + varimax) và trích Factor Scores

Giống `run_factor_analysis()` trong code:

- `FactorAnalyzer(n_factors=n_factors, method="principal", rotation="varimax")`
- Fit trên dữ liệu chuẩn hoá
- Lấy:
  - `loadings_` (ma trận factor loadings)
  - `transform()` (factor scores cho từng quan sát)

Bạn có thể **override** `n_factors` ở cell trước nếu muốn (ví dụ cố định = 3 để khớp đề tài).

In [ ]:
# Nếu muốn cố định số nhân tố theo đề tài, mở comment dòng dưới
# n_factors = 3

fa = FactorAnalyzer(n_factors=n_factors, method="principal", rotation="varimax")
fa.fit(fa_std)

scores = fa.transform(fa_std)
factor_cols = [f"Factor{i+1}" for i in range(n_factors)]
scores_df = pd.DataFrame(scores, index=fa_std.index, columns=factor_cols)

print("Scores shape:", scores_df.shape)
scores_df.head()

## 7. Factor loadings, heatmap, và mức giải thích phương sai

- Xuất `factor_loadings.csv`
- Vẽ heatmap (Varimax)
- Xem tổng phương sai giải thích theo nhân tố (Factor Variance)

In [ ]:
# Factor loadings
loadings = fa.loadings_
loadings_df = pd.DataFrame(loadings, index=fa_std.columns, columns=factor_cols).round(4)

loadings_path = TABLES_DIR / "factor_loadings.csv"
loadings_df.to_csv(loadings_path)
print("Saved:", loadings_path)

display(loadings_df)

# Heatmap (giống save_factor_loadings)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    loadings_df.round(3),
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    center=0,
    vmin=-0.8,
    vmax=0.8,
    ax=ax,
    annot_kws={"size": 9},
)
ax.set_title("Factor Loadings (Varimax Rotation)", fontsize=14)
plt.xticks(rotation=0)
plt.tight_layout()

heatmap_path = FIG_DIR / "factor_loadings_heatmap.png"
plt.savefig(heatmap_path, dpi=300, bbox_inches="tight")
plt.show()
print("Saved:", heatmap_path)

# Variance explained (table chuẩn để đưa vào báo cáo)
ss_loadings, prop_var, cum_var = fa.get_factor_variance()
variance_df = pd.DataFrame(
    {
        "SS Loadings": ss_loadings,
        "Proportion Var": prop_var,
        "Proportion Var (%)": prop_var * 100,
        "Cumulative Var": cum_var,
        "Cumulative Var (%)": cum_var * 100,
    },
    index=factor_cols,
).round(4)

variance_path = TABLES_DIR / "factor_variance_explained.csv"
variance_df.to_csv(variance_path)
print("Saved:", variance_path)

display(variance_df)

## 8. Diễn giải nhân tố (tự động) và lưu `fa_data.csv`

Notebook sẽ:

- Tạo phần **diễn giải** (top biến theo |loading| > 0.4) tương tự `save_factor_loadings()`.
- Ghép `Factor1..k` vào dữ liệu đầy đủ và lưu `data/processed/fa_data.csv`.

Lưu ý: `factor_analysis.py` ghép factor scores theo index thời gian. Ở đây ta dùng `fa_df_nonan` (đã dropna cho FA), nên khi concat vào `df` sẽ tạo **NaN** ở những timestamp bị drop. Điều này là bình thường; về sau các bước có thể resample/mean hoặc dropna phù hợp.

In [ ]:
# Diễn giải từng factor: liệt kê TOP (+) và TOP (−) loadings
thresh = 0.4
var_desc = {
    "PM10": "bụi mịn PM10",
    "SO2": "lưu huỳnh đioxit",
    "NO2": "nitơ đioxit",
    "CO": "cacbon monoxit",
    "O3": "ozon",
    "TEMP": "nhiệt độ",
    "PRES": "áp suất",
    "DEWP": "điểm sương",
    "RAIN": "lượng mưa",
    "WSPM": "tốc độ gió",
}

def _fmt_vars(vars_list, col):
    return ", ".join(f"{v} ({col[v]:+.2f})" for v in vars_list)

lines = [
    "FACTOR INTERPRETATION (Varimax)",
    "=" * 60,
    "",
    "Quy ước:",
    "- |loading| >= 0.4: đóng góp đáng kể",
    "- Top (+): các biến tương quan cùng chiều với nhân tố",
    "- Top (−): các biến tương quan ngược chiều với nhân tố",
    "",
]

factor_summary_rows = []

for fc in factor_cols:
    col = loadings_df[fc]
    pos = col.sort_values(ascending=False)
    neg = col.sort_values(ascending=True)

    top_pos = [v for v in pos.index if pos[v] >= thresh][:5]
    top_neg = [v for v in neg.index if neg[v] <= -thresh][:5]

    if not top_pos:
        top_pos = list(pos.index[:3])
    if not top_neg:
        top_neg = list(neg.index[:3])

    # Gợi ý nhãn dựa theo nhóm biến xuất hiện nhiều nhất
    pollution = {"PM10", "SO2", "NO2", "CO"}
    meteo = {"TEMP", "PRES", "DEWP", "RAIN"}
    diffusion = {"WSPM"}

    top_union = set(top_pos + top_neg)
    label_hits = {
        "Emission/Pollution": len(top_union & pollution),
        "Meteorology": len(top_union & meteo),
        "Diffusion": len(top_union & diffusion),
        "Photochemical (O3)": 1 if "O3" in top_union else 0,
    }
    suggested = max(label_hits, key=label_hits.get)

    lines.append(f"{fc} (suggested theme: {suggested})")
    lines.append("-" * 60)
    lines.append(f"Top (+): {_fmt_vars(top_pos, col)}")
    lines.append(f"Top (−): {_fmt_vars(top_neg, col)}")
    lines.append("Diễn giải (+): " + ", ".join(var_desc.get(v, v) for v in top_pos))
    lines.append("Diễn giải (−): " + ", ".join(var_desc.get(v, v) for v in top_neg))
    lines.append("")

    factor_summary_rows.append(
        {
            "Factor": fc,
            "SuggestedTheme": suggested,
            "TopPos": "; ".join(top_pos),
            "TopNeg": "; ".join(top_neg),
        }
    )

interp_path = TABLES_DIR / "factor_interpretation.txt"
with open(interp_path, "w", encoding="utf-8") as f:
    f.write("\n".join(lines))
print("Saved:", interp_path)

pd.DataFrame(factor_summary_rows)

## 9. Factor scores theo thời gian, tương quan với PM2.5, và dataset chuẩn bị cho SARIMA

Mục tiêu phần này:

- Vẽ **Factor1..k theo thời gian** (hourly và/hoặc daily).
- Tính **correlation** giữa `PM2.5` và các factor (trên chuỗi daily để khớp SARIMA).
- Xuất dataset **daily** gồm `PM2.5` + `Factor1..k` để dùng cho SARIMA (exogenous variables).

In [ ]:
# Ghép factor scores vào full dataframe (không đứt gãy)
# scores_df đang có index = fa_std.index (toàn bộ timeline vì ta đã interpolate)
result = pd.concat([df, scores_df], axis=1)

# Check nhanh missing sau khi ghép
print("Missing in factors (after concat):")
display(result[factor_cols].isna().sum())

# Save hourly-level fa_data.csv (giống pipeline)
fa_out_path = PROCESSED_DIR / "fa_data.csv"
result.to_csv(fa_out_path)
print("Saved:", fa_out_path)

result[["PM2.5"] + factor_cols].head()

In [ ]:
# Plot factor scores theo time (hourly) - lấy rolling để dễ nhìn
plot_df = result[factor_cols].copy()

# Downsample về daily mean cho plot (đỡ nặng và khớp SARIMA)
plot_daily = plot_df.resample("D").mean()

fig, ax = plt.subplots(figsize=(12, 5))
for c in factor_cols:
    ax.plot(plot_daily.index, plot_daily[c], linewidth=1.0, label=c)
ax.set_title("Factor Scores over Time (Daily Mean)")
ax.set_xlabel("Date")
ax.set_ylabel("Score")
ax.grid(True, alpha=0.3)
ax.legend(ncol=min(3, len(factor_cols)))
plt.tight_layout()

scores_plot_path = FIG_DIR / "factor_scores_time_series_daily.png"
plt.savefig(scores_plot_path, dpi=300, bbox_inches="tight")
plt.show()
print("Saved:", scores_plot_path)

In [ ]:
# Correlation: Factor vs PM2.5 (daily) để khớp mô hình SARIMA daily
if "PM2.5" not in result.columns:
    raise KeyError("Missing column PM2.5 in cleaned dataset")

pm25_daily = result["PM2.5"].astype(float).resample("D").mean()
factors_daily = result[factor_cols].resample("D").mean()

corr_df = pd.concat([pm25_daily.rename("PM2.5"), factors_daily], axis=1).dropna()

corr_matrix = corr_df.corr()

# Bảng tương quan PM2.5 với các factor
corr_pm25 = corr_matrix.loc[factor_cols, "PM2.5"].sort_values(key=lambda s: s.abs(), ascending=False)
print("Correlation (daily) between Factors and PM2.5:")
display(corr_pm25.to_frame(name="corr_with_PM2.5"))

corr_out = TABLES_DIR / "corr_factors_vs_pm25_daily.csv"
corr_pm25.to_csv(corr_out)
print("Saved:", corr_out)

# Heatmap toàn bộ
fig, ax = plt.subplots(figsize=(6, 4))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="RdBu_r", center=0, ax=ax)
ax.set_title("Correlation: PM2.5 & Factors (Daily)")
plt.tight_layout()

corr_fig = FIG_DIR / "corr_pm25_factors_daily.png"
plt.savefig(corr_fig, dpi=300, bbox_inches="tight")
plt.show()
print("Saved:", corr_fig)

In [ ]:
# Dataset chuẩn bị cho SARIMA: daily PM2.5 + daily factors (exog)
# Đây là input trực tiếp cho Step 3 (SARIMA) ở tần suất ngày
sarima_daily = pd.concat([pm25_daily.rename("PM2.5"), factors_daily], axis=1)

# Nội suy để đảm bảo daily series liên tục (nếu có ngày thiếu do resample)
for col in sarima_daily.columns:
    if sarima_daily[col].isna().any():
        sarima_daily[col] = sarima_daily[col].interpolate(method="time")

sarima_daily = sarima_daily.ffill().bfill()

print("SARIMA daily input shape:", sarima_daily.shape)
print("Index inferred freq:", pd.infer_freq(sarima_daily.index))

sarima_path = PROCESSED_DIR / "sarima_input_daily.csv"
sarima_daily.to_csv(sarima_path)
print("Saved:", sarima_path)

sarima_daily.head()